In [2]:
import sys

scope_repo = "/Users/sergivela/Documents/SCOPE/Program/Scope"
sys.path.insert(0, f"{scope_repo}/core/src")
sys.path.insert(0, f"{scope_repo}/sco/src")

import scope
import scope_sco

In [3]:
import os
import csv
import matplotlib.pyplot as plt

In [4]:
%run ./5a-Core_Ring_Map_Functions.ipynb

In [7]:
figures_path    = "/Users/sergivela/Documents/SCOPE/Database_SCO/Manuscript/figures"
env_path        = "/Users/sergivela/Documents/SCOPE/Database_SCO/Data/scope_env_atlas.npy"
root_path       = "/Users/sergivela/Documents/SCOPE/Database_SCO/Analysis/1-Analyse_Database/"
outputs_path    = "/Users/sergivela/Documents/SCOPE/Database_SCO/Analysis/1-Analyse_Database/Outputs/"
project_path    = "/Users/sergivela/Documents/SCOPE/Database_SCO/"
env             = scope.read_write.load_binary(env_path)

## Loads Systems

### All systems and Finished ones

In [8]:
finished_csv_path     = outputs_path + "finished_systems.csv"
not_finished_csv_path = outputs_path + "not_finished_systems.csv"

finished_systems = []
not_finished = []
with open(finished_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sys = scope.read_write.load_binary(f"{project_path}/{row['path']}")
        sys.set_paths_from_environment(env)
        finished_systems.append(sys)

with open(not_finished_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sys = scope.read_write.load_binary(f"{project_path}/{row['path']}")
        sys.set_paths_from_environment(env)
        not_finished.append(sys)

## Breakdown of Crystals

In [ ]:
overwrite                   = True
breakdown_csv1_path         = outputs_path + "breakdown_finished_crystals.csv"
breakdown_finished_rows     = []

In [ ]:
num_crystals_HS = 0
num_crystals_LS = 0
num_crystals_other = 0
highest = 0

for sys in finished_systems:
    relative_path       = f"/Data/2-Systems/{sys.refcode}/{sys.refcode}.npy"
    # Counts number of original crystals
    ncrystals          = 0
    num_crystals_HS    = 0
    num_crystals_LS    = 0
    num_crystals_other = 0
    temp_HS_diff       = []
    temp_LS_diff       = []
    for sou in sys.sources:
        if sou.object_type == "cell" and "ref" not in sou.name:
            ncrystals += 1
            if hasattr(sou,"phase"):
                if   sou.phase == "HS": 
                    num_crystals_HS    += 1
                    temp_HS_diff.append(sou.cif.diff_temp)
                elif sou.phase == "LS": 
                    num_crystals_LS    += 1
                    temp_LS_diff.append(sou.cif.diff_temp)
                else:                   
                    num_crystals_other += 1
    # Checks if maximum
    if ncrystals > highest: 
        highest = ncrystals
        highest_refcode = sys.refcode
    breakdown_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "status": "finished", "ncrystals": ncrystals, "n_HS": num_crystals_HS, "n_LS": num_crystals_LS, "n_other": num_crystals_other, "temp_HS_diff": temp_HS_diff, "temp_LS_diff": temp_LS_diff}) 

with open(breakdown_csv1_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["refcode", "path", "status", "ncrystals", "n_HS", "n_LS", "n_other", "temp_HS_diff", "temp_LS_diff"])
    writer.writeheader()
    writer.writerows(breakdown_finished_rows)

print(count_csv_entries(breakdown_csv1_path, "status", "finished"))
print(sum_csv_entries(breakdown_csv1_path, "ncrystals"))
print(sum_csv_entries(breakdown_csv1_path, "n_HS"))
print(sum_csv_entries(breakdown_csv1_path, "n_LS"))
print(sum_csv_entries(breakdown_csv1_path, "n_other"))

721
1148
474
623
51


In [ ]:
breakdown_csv2_path         = outputs_path + "breakdown_non_finished_crystals.csv"
breakdown_non_finished_rows = []
highest = 0

for sys in not_finished:
    relative_path       = f"/Data/2-Systems/{sys.refcode}/{sys.refcode}.npy"
    # Counts number of original crystals
    ncrystals          = 0
    num_crystals_HS    = 0
    num_crystals_LS    = 0
    num_crystals_other = 0
    temp_HS_diff       = []
    temp_LS_diff       = []
    for sou in sys.sources:
        if sou.object_type == "cell" and "ref" not in sou.name:
            ncrystals += 1
            if hasattr(sou,"phase"):
                if   sou.phase == "HS": 
                    num_crystals_HS    += 1
                    temp_HS_diff.append(sou.cif.diff_temp)
                elif sou.phase == "LS": 
                    num_crystals_LS    += 1
                    temp_LS_diff.append(sou.cif.diff_temp)
                else:                   num_crystals_other += 1
    # Checks if maximum
    if ncrystals > highest: 
        highest = ncrystals
        highest_refcode = sys.refcode
    breakdown_non_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "status": "finished", "ncrystals": ncrystals, "n_HS": num_crystals_HS, "n_LS": num_crystals_LS, "n_other": num_crystals_other, "temp_HS_diff": temp_HS_diff, "temp_LS_diff": temp_LS_diff}) 

with open(breakdown_csv2_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["refcode", "path", "status", "ncrystals", "n_HS", "n_LS", "n_other", "temp_HS_diff", "temp_LS_diff"])
    writer.writeheader()
    writer.writerows(breakdown_non_finished_rows)

print(count_csv_entries(breakdown_csv2_path, "status", "not_finished"))
print(sum_csv_entries(breakdown_csv2_path, "ncrystals"))
print(sum_csv_entries(breakdown_csv2_path, "n_HS"))
print(sum_csv_entries(breakdown_csv2_path, "n_LS"))
print(sum_csv_entries(breakdown_csv2_path, "n_other"))

0
23
4
18
1


## Evaluate which systems have both LS and HS crystals, and which ones are likely to be due to Thermal or LIESST experiments

In [15]:
count = 0
for item in filter_rows(breakdown_csv1_path, lambda row: int(row["n_HS"]) > 0 and int(row["n_LS"]) > 0):
    print(f"{item["refcode"]:<10} {item["n_HS"]:<10} {item["n_LS"]:<10} {item["temp_HS_diff"]:<40} {item["temp_LS_diff"]:<40}")
    count += 1
print(count)

ABITEM     1          1          [240.0]                                  [104.0]                                 
ACAHUJ     1          1          [292.0]                                  [116.0]                                 
ADAQUU     1          1          [448.0]                                  [150.0]                                 
ADAREF     1          1          [250.0]                                  [220.0]                                 
AFANEB     1          1          [200.0]                                  [90.0]                                  
AFOYUS     2          1          [293.0, 353.0]                           [123.0]                                 
AKENAF     1          1          [296.2]                                  [88.2]                                  
ATAVEY     1          1          [250.0]                                  [100.0]                                 
BAKGUR     1          1          [296.1]                                  [110.1

In [16]:
print(f"{'Refcode':<10} {'HS Phase':<10} {'LS Phase':<10} {'HS Diff Temp':<15} {'LS Diff Temp':<15}")
print("-" * 60)
for sys in both:
    print(f"{sys.refcode_wo_digits:<10} {getattr(sys.HS_ref_crys, 'phase', ''):<10} {getattr(sys.LS_ref_crys, 'phase', ''):<10} {getattr(sys.HS_ref_crys, 'diff_temp', '')!s:<15} {getattr(sys.LS_ref_crys, 'diff_temp', '')!s:<15}")

Refcode    HS Phase   LS Phase   HS Diff Temp    LS Diff Temp   
------------------------------------------------------------


NameError: name 'both' is not defined

# BELOW HASN'T BEEN EXECUTED

### Ligands

In [6]:
# Exporting ligands from the crystal. It is better to extract them from the state information, which is what I do below
list_of_ligands = []
for sys in finished_systems:
    added = False
    goodmol = False
    for mol in sys.HS_ref_crys.list_of_molecules:
        if not goodmol:
            for lig in mol.ligands:
                if hasattr(lig,"rdkit_obj"): 
                    lig.fix_rdkit_obj()
                    lig.set_smiles_from_rdkit_obj()
                    list_of_ligands.append(lig)
                    goodmol = True
                    added = True
    if not added: print(sys.refcode)
print(len(list_of_ligands), "ligands found")

2027 ligands found


In [7]:
skiplist = []
duplicates = []
unique_ligands = []
#smiles = []
#mhfps = []
for idx, lig1 in enumerate(list_of_ligands):
    if idx not in skiplist:
        frag1 = lig1.check_fragmentation()
        if not frag1: 
            lig1.scope_FeN_ID = idx
            unique_ligands.append(lig1)
        else:
            skiplist.append(idx)
            if hasattr(lig1,"parents"):
                print(lig1.formula, lig1.get_parent("cell").refcode, "fragmented")
            else:
                print(idx, lig1.parent.formula)
#        smiles.append(lig1.smiles)
#        mhfps.append(lig1.mhfp)
        for jdx, lig2 in enumerate(list_of_ligands):
            if jdx > idx and jdx not in skiplist:
                issame = compare_species(lig1, lig2)
                if issame:
                    skiplist.append(jdx)
                    duplicates.append([lig1, lig2])
print(len(unique_ligands), "unique ligands found")

#was:
#H44-C36-N4 CPOEFE10 fragmented
#H20-C15-N4 NUMBEC02 fragmented
#H33-C24-N7 WOJLUC fragmented
#555 unique ligands found

H44-C36-N4 CPOEFE10 fragmented
H20-C15-N4 NUMBEC02 fragmented
H33-C24-N7 WOJLUC fragmented
556 unique ligands found


In [ ]:
print(len(list_of_ligands)) # was 1975, then 3093, then 2018, when I only searched in HS_ref_crys
print(len(unique_ligands))  # was 551, then 556, then 549, then 555 

2027
556


In [17]:
list_of_ligands_ids = []
for sys in finished_systems:
    tmp = []
    for idx, lig in enumerate(sys.HS_ref_mol.ligands):
        found = False
        for jdx, ulig in enumerate(unique_ligands):
            if not found:
                issame = compare_species(lig, ulig)
                if issame:
                    found = True
                    tmp.append(list([sys.refcode,idx,jdx]))
        if not found: 
            if sys.refcode == "QAHVIE" and lig.formula == "H9-C12-N6": ## Corrects QAHVIE ligand, which is fragmented in crys.cell.moleclist.ligands object 
                tmp.append(list([sys.refcode,idx,58]))
            elif sys.refcode == "WOJLUC" and lig.formula == "H30-C24-N7": 
                tmp.append(list([sys.refcode,idx,58]))
            
            else:
                print(sys.refcode, lig.formula, "not found")
    list_of_ligands_ids.append(tmp)

QAHVIE H10-C12-N6 not found
QAHVIE H10-C12-N6 not found
WOJLUC H33-C24-N7 not found


In [18]:
save_binary(list_of_ligands,"/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/all_ligands.npy")
save_binary(unique_ligands,"/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/unique_ligands.npy")
save_binary(list_of_ligands_ids,"/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/list_of_ligands_ids.npy")

### Intentant debuguejar casos aqui sota

In [23]:
for idx, ulig in enumerate(unique_ligands):
    print(ulig.formula, ulig.get_denticity(), ulig.get_connected_metals()[0].get_coord_sphere_formula())

C-N-S 1 N6
H18-C18-N4 4 N6
H11-C11-N3-O 2 N6
H24-C23-N8 4 N6
H28-C25-N8 4 N6
H24-C18-N2 2 N6
H28-C26-N6 6 N6
H44-C34-N6 6 N6
H13-C13-N5 3 N6
C-N-Se 1 N6
H11-C14-N3 2 N6
H3-C2-N 1 N6
H20-C16-N6-O4 4 N6
H16-C20-N4-S4 2 N6
H12-C9-N2 2 N6
H20-B-C16-N8 3 N6
H20-C17-N6-O2 3 N6
H17-C14-N7 3 N6
H19-C23-N3-O2 3 N6
H13-C19-N5-S4 3 N6
H8-C6-N2 2 N6
H5-C5-N 1 N6
H11-C14-N2-O2 2 N6
H16-C16-N4-O 2 N6
H16-C17-N6 3 N6
H32-C36-N6 6 N6
H10-C17-N4-S 2 N6
H8-B-C6-N4 2 N6
H10-C10-N4 2 N6
H3-N 1 N6
H18-C22-N4 4 N6
H9-C11-N5 3 N6
H19-C15-N3-O2 3 N6
H4-C3-N2 1 N6
H8-C12-N6 3 N6
H8-C12-N2 2 N6
H13-C8-N3 2 N6
H24-C24-N6 6 N6
H7-B-C6-N9 3 N6
H21-C16-N3 2 N6
H18-C14-N4 4 N6
H36-C32-N4 4 N6
H10-C14-N2-S2 2 N6
H14-C18-N5-Cl 2 N6
H14-C18-N5-Br 2 N6
H14-C12-N4 3 N6
H10-C12-N4 3 N6
H6-C4-N2 1 N6
H32-B2-C32-N4-O4 4 N6
H8-C11-N4 2 N6
H38-C23-N6 6 N6
H8-C9-N2 2 N6
H27-C20-N5 3 N6
H15-C19-N5 2 N6
H10-C9-N4-O 2 N6
H16-C24-N2 2 N6
H20-C16-N4 4 N6
C4-N3 1 N6
H10-C12-N6 2 N6
H3-C8-N4-O 1 N6
H5-C9-N4-O 1 N6
H22-C30-N7-O2 3 N6


In [26]:
for idx, ulig in enumerate(unique_ligands):
    if ulig.formula == "H10-C12-N6": print(idx, ulig.smiles)
    elif ulig.formula == "H33-C24-N7": print(idx, ulig.smiles)
    elif "C24" in ulig.formula and "N7" in ulig.formula: print(ulig.formula, ulig.smiles, idx)

58 [H]c1nc(-c2nnc(-c3nc([H])c([H])c([H])c3[H])n2N([H])[H])c([H])c([H])c1[H]
H27-C24-N7 [H]C(=NC([H])([H])C([H])([H])N(C([H])([H])C([H])([H])N=C([H])c1nc([H])c([H])c([H])c1[H])C([H])([H])C([H])([H])N=C([H])c1nc([H])c([H])c([H])c1[H])c1nc([H])c([H])c([H])c1[H] 239
H24-C24-N7-F3 [H]C(=NC([H])([H])C([H])([H])N(C([H])([H])C([H])([H])N=C([H])c1nc([H])c(F)c([H])c1[H])C([H])([H])C([H])([H])N=C([H])c1nc([H])c(F)c([H])c1[H])c1nc([H])c(F)c([H])c1[H] 435


In [14]:
for sys in finished_systems:
    if sys.refcode == "QAHVIE":
        for crys in sys.crystals:
            for mol in crys.cell.moleclist: 
                for lig in mol.ligands:
                    if hasattr(lig,"rdkit_obj"): 
                        if lig.formula == "H10-C12-N6": print(lig.formula)
        #for idx, lig in enumerate(sys.HS_ref_mol.ligands):
        #    print(lig.formula)

H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6
H10-C12-N6